# Coffee price by time of day: Tukey's test

**Two-Way Analysis of Variance** · a worked example from the StatNex teaching guide

**The story.** A campus coffee cart tried three prices for a regular coffee (\$2.50, \$3.50, \$4.50) at four times of day. Each combination was tried on three days, and cups sold were counted.

**The question.** Does the effect of price depend on the time of day?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `05_anova/two_way_anova.csv` (36 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Price,Time of day,Cups sold
$2.50,Morning,43.0
$2.50,Morning,39.0
$2.50,Morning,41.0
$2.50,Lunch,32.0
$2.50,Lunch,35.0
$2.50,Lunch,34.0
$2.50,Afternoon,21.0
$2.50,Afternoon,27.0
$2.50,Afternoon,27.0
$2.50,Evening,15.0
$2.50,Evening,13.0
$2.50,Evening,14.0
$3.50,Morning,29.0
$3.50,Morning,31.0
$3.50,Morning,30.0
$3.50,Lunch,27.0
$3.50,Lunch,23.0
$3.50,Lunch,20.0
$3.50,Afternoon,22.0
$3.50,Afternoon,26.0
$3.50,Afternoon,26.0
$3.50,Evening,12.0
$3.50,Evening,11.0
$3.50,Evening,12.0
$4.50,Morning,26.0
$4.50,Morning,28.0
$4.50,Morning,25.0
$4.50,Lunch,22.0
$4.50,Lunch,23.0
$4.50,Lunch,24.0
$4.50,Afternoon,20.0
$4.50,Afternoon,20.0
$4.50,Afternoon,19.0
$4.50,Evening,10.0
$4.50,Evening,7.0
$4.50,Evening,6.0

These cells save `eta_squared_interval.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile eta_squared_interval.py
"""The confidence interval for partial eta squared, from the noncentral F distribution.

SciPy has the noncentral F (stats.ncf) but no function that inverts it for an interval,
so the inversion is written out (Steiger 2004). The interval is found for the
noncentrality λ and converted to partial eta squared as λ / (λ + df1 + df2 + 1).
"""

import numpy as np
from scipy import optimize, stats


def _limit(f, df1, df2, q):
    """The λ ≥ 0 under which the observed F sits at the q quantile of F(df1, df2, λ).

    0 when even λ = 0 cannot put it there: an F too small for the interval to leave 0.
    The cdf at F falls as λ rises, so the bracket opens outwards from F·df1, doubling.
    """
    def gap(lam):
        cdf = stats.f.cdf(f, df1, df2) if lam == 0 else stats.ncf.cdf(f, df1, df2, lam)
        return cdf - q

    if gap(0.0) <= 0:
        return 0.0
    start = max(f * df1, 1.0)
    step = 1.0
    while gap(max(start - step, 0.0)) <= 0:
        step *= 2
    low = max(start - step, 0.0)
    step = 1.0
    while gap(start + step) >= 0:
        step *= 2
    return optimize.brentq(gap, low, start + step, xtol=1e-14,
                           rtol=4 * np.finfo(float).eps, maxiter=500)


def partial_eta_squared_interval(f, df1, df2, level):
    """(lower, upper): the level interval for partial eta squared (Steiger 2004)."""
    total = df1 + df2 + 1
    lower = _limit(f, df1, df2, (1 + level) / 2)
    upper = _limit(f, df1, df2, (1 - level) / 2)
    return lower / (lower + total), upper / (upper + total)

## Libraries

In [ ]:
from itertools import combinations

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.anova import anova_lm

pd.set_option("display.width", 120)

## The data

In [ ]:
# One row per observation: the response and the two factors. Rows with a blank are
# dropped. Levels keep the order they first appear in; every combination of levels
# needs data, and some cell needs at least two observations.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
response = "Cups sold"
column_factor = "Price"
row_factor = "Time of day"
frame = pd.DataFrame({"y": pd.to_numeric(data[response], errors="coerce"),
                      "A": data[column_factor], "B": data[row_factor]}).dropna()
column_levels = list(frame["A"].unique())
row_levels = list(frame["B"].unique())
frame["A"] = pd.Categorical(frame["A"], categories=column_levels)
frame["B"] = pd.Categorical(frame["B"], categories=row_levels)

## Descriptive statistics

In [ ]:
# Each level of the column factor (A), of the row factor (B), then each cell.
def describe(by):
    table = frame.groupby(by, observed=True)["y"].agg(n="count", mean="mean", SD="std")
    table.index = [" * ".join(map(str, np.atleast_1d(key))) for key in table.index]
    table["SE"] = table["SD"] / np.sqrt(table["n"])
    return table.assign(**{"mean - SD": table["mean"] - table["SD"],
                           "mean + SD": table["mean"] + table["SD"],
                           "mean - SE": table["mean"] - table["SE"],
                           "mean + SE": table["mean"] + table["SE"]})[
        ["n", "mean", "SD", "mean - SD", "mean + SD", "SE", "mean - SE", "mean + SE"]]


descriptive = pd.concat([describe("A"), describe("B"), describe(["A", "B"])],
                        keys=["A", "B", "A * B"])
print(descriptive.to_string())

## Test for homogeneity of variance

In [ ]:
# Levene's original form across the cells, on deviations from each cell's mean
# (center="mean"; SciPy's default is the median).
cells = [group["y"].to_numpy() for _, group in frame.groupby(["A", "B"], observed=True)]
levene = stats.levene(*cells, center="mean")
homogeneity = pd.DataFrame([[levene.statistic, len(cells) - 1, len(frame) - len(cells),
                             levene.pvalue]], columns=["F", "df1", "df2", "P"])
print()
print(homogeneity.to_string(index=False))

## Analysis of variance table

In [ ]:
# The effects use sum-to-zero coding (C(..., Sum)). With equal cell sizes the three
# types of sums of squares coincide with the textbook decomposition; with unequal
# cells they differ in what each main effect is adjusted for. The interaction is
# entered last in every type, so its line is the same in all three.
fit = smf.ols("y ~ C(A, Sum) * C(B, Sum)", data=frame).fit()
# Type III: each effect measured after the other two, so the table depends neither
# on the order of the factors nor on which level is listed last.
table = anova_lm(fit, typ=3).drop(index="Intercept")
ss_error, df_error = table.loc["Residual", "sum_sq"], table.loc["Residual", "df"]
ms_error = ss_error / df_error
anova = pd.DataFrame({"SS": table["sum_sq"].to_numpy(), "df": table["df"].to_numpy()},
                     index=["A (column)", "B (row)", "A * B", "Error"])
anova["MS"] = anova["SS"] / anova["df"]
anova["F"] = anova["MS"] / ms_error
anova["P"] = stats.f.sf(anova["F"], anova["df"], df_error)
anova.loc["Error", ["F", "P"]] = np.nan
anova.loc["Total"] = [((frame["y"] - frame["y"].mean()) ** 2).sum(), len(frame) - 1,
                      np.nan, np.nan, np.nan]
print()
print(anova)

columns = ["level 1", "level 2", "mean 1", "mean 2", "difference", "SE", "statistic", "P"]


# One family of pairwise comparisons: every pair of the given levels, statistic =
# |difference| / SE with SE = sqrt(MS error x (1/n1 + 1/n2)), on the error df of the
# whole model. The family is what an adjustment counts -- one factor's levels, or the
# levels of one factor inside one level of the other.
def pair_rows(values, levels):
    rows = []
    for a, b in combinations(levels, 2):
        x, z = values[a], values[b]
        se = np.sqrt(ms_error * (1 / len(x) + 1 / len(z)))
        difference = abs(x.mean() - z.mean())
        rows.append([a, b, x.mean(), z.mean(), difference, se, difference / se])
    t, k = np.array([row[-1] for row in rows]), len(levels)
    # Tukey: the studentised range, q = statistic x sqrt(2) (Tukey-Kramer when the
    # sizes differ).
    p = stats.studentized_range.sf(t * np.sqrt(2), k, df_error)
    return [[*row, p_value] for row, p_value in zip(rows, p)]

## Multiple comparison

In [ ]:
# Every pair of levels of each factor, on the plain level means and the pooled error;
# each factor is a family of its own.
pairs = pd.DataFrame(
    pair_rows({a: g["y"].to_numpy() for a, g in frame.groupby("A", observed=True)}, column_levels)
    + pair_rows({b: g["y"].to_numpy() for b, g in frame.groupby("B", observed=True)}, row_levels),
    columns=columns)
print()
print(pairs.to_string(index=False))

## Least-squares means

In [ ]:
# The mean each level would have if every cell weighed the same: the average of its
# cell means, with SE = sqrt(MS error x sum(1 / n of each cell)) / number of cells.
# With equal cell sizes this is the plain mean.
confidence = 95 / 100
critical = stats.t.ppf(0.5 + confidence / 2, df_error)
cell_mean = frame.pivot_table("y", "A", "B", aggfunc="mean", observed=True)
cell_n = frame.pivot_table("y", "A", "B", aggfunc="count", observed=True)
rows = []
for c in column_levels:
    se = np.sqrt(ms_error * (1 / cell_n.loc[c]).sum()) / len(row_levels)
    rows.append([c, cell_n.loc[c].sum(), cell_mean.loc[c].mean(), se])
for r in row_levels:
    se = np.sqrt(ms_error * (1 / cell_n[r]).sum()) / len(column_levels)
    rows.append([r, cell_n[r].sum(), cell_mean[r].mean(), se])
for c in column_levels:
    for r in row_levels:
        rows.append([f"{c} * {r}", cell_n.loc[c, r], cell_mean.loc[c, r],
                     np.sqrt(ms_error / cell_n.loc[c, r])])
lsmeans = pd.DataFrame(rows, columns=["level", "n", "LS mean", "SE"])
lsmeans["lower"] = lsmeans["LS mean"] - critical * lsmeans["SE"]
lsmeans["upper"] = lsmeans["LS mean"] + critical * lsmeans["SE"]
print()
print(lsmeans.to_string(index=False))

## Simple main effects

In [ ]:
# One factor tested separately at each level of the other. The error is the pooled
# one from the whole model -- not each slice's own -- so it keeps all its degrees of
# freedom. First the column factor within each row level, then the reverse.
simple = {}
for outer, inner, means, counts in (("B", "A", cell_mean.T, cell_n.T),
                                    ("A", "B", cell_mean, cell_n)):
    tests, pairs_within = [], []
    for level in means.index:
        m, k = means.loc[level], counts.loc[level]
        ss = (k * (m - (k * m).sum() / k.sum()) ** 2).sum()
        df = len(m) - 1
        f_value = (ss / df) / ms_error
        tests += [[level, inner, ss, df, ss / df, f_value, stats.f.sf(f_value, df, df_error)],
                  [level, "Error", ss_error, df_error, ms_error, np.nan, np.nan]]
        # The pairs inside this one level are a family of their own.
        values = {i: g["y"].to_numpy()
                  for i, g in frame[frame[outer] == level].groupby(inner, observed=True)}
        pairs_within += [[level, *row] for row in pair_rows(values, list(m.index))]
    simple[outer] = {
        "means": means, "n": counts,
        "replications": pd.DataFrame([[counts.to_numpy().min(), counts.to_numpy().max()]],
                                     columns=["min", "max"]),
        "tests": pd.DataFrame(tests, columns=["level", "source", "SS", "df", "MS", "F", "P"]),
        "pairs": pd.DataFrame(pairs_within, columns=["level", *columns]),
    }
    print(f"\n{inner} within each level of {outer}: means, n, and the tests")
    print(means)
    print(counts)
    print(simple[outer]["tests"].to_string(index=False))
    print(simple[outer]["pairs"].to_string(index=False))

## Effect sizes

In [ ]:
# For each line of the table: eta² = SS / SS total; partial eta² = SS / (SS + SS error),
# the effect against the error alone; omega² = (SS - df·MS error) / (SS total + MS error),
# which removes eta²'s upward bias (Hays 1963); and partial omega² = (SS - df·MS error)
# / (SS + (N - df)·MS error) (Olejnik & Algina 2003). Partial eta² carries the interval,
# at the confidence level above, that inverts the noncentral F at the line's F (Steiger
# 2004), written out in eta_squared_interval.py.
from eta_squared_interval import partial_eta_squared_interval


def size(value):
    """Cohen's (1988) reading of an eta²: 0.01 small, 0.06 medium, 0.14 large."""
    return ["negligible", "small", "medium", "large"][sum(value >= s for s in (0.01, 0.06, 0.14))]


ss_total, n_total = anova.loc["Total", "SS"], len(frame)
rows = []
for name in ["A (column)", "B (row)", "A * B"]:
    ss, df, f_value = anova.loc[name, ["SS", "df", "F"]]
    lower, upper = partial_eta_squared_interval(f_value, df, df_error, confidence)
    rows += [[name, "eta²", ss / ss_total, np.nan, np.nan],
             [name, "partial eta²", ss / (ss + ss_error), lower, upper],
             [name, "omega²", (ss - df * ms_error) / (ss_total + ms_error), np.nan, np.nan],
             [name, "partial omega²", (ss - df * ms_error) / (ss + (n_total - df) * ms_error),
              np.nan, np.nan]]
effect_sizes = pd.DataFrame(rows, columns=["effect", "measure", "value", "lower", "upper"])
effect_sizes["size"] = effect_sizes["value"].map(size)
print()
print(effect_sizes.to_string(index=False))

## Normality of the residuals: the Shapiro-Wilk test

In [ ]:
# The residuals are each value minus its cell's mean -- the errors the ANOVA assumes
# normal -- rather than the values, which mix the cells' different means.
shapiro = stats.shapiro(fit.resid)
normality = [len(fit.resid), shapiro.statistic, shapiro.pvalue]
print(f"\nShapiro-Wilk on the residuals: n = {len(fit.resid)}, W = {shapiro.statistic:.4f}, "
      f"P = {shapiro.pvalue:.4g}")

## What the result says

Two-way analysis of variance tests price, time of day and their interaction. An interaction means the effect of price depends on the time of day. It has F = 6.77 and a P value far below 0.01. F compares an effect with the noise, and the P value is the chance of an effect at least this large if there were none.

So price is read separately for each time of day. Tukey's test compares the prices and corrects for comparing three pairs. In the morning, going from \$2.50 to \$3.50 drops sales from 41 to 30 cups. In the afternoon the same step goes from 25 to 24.7 cups, with P = 0.977: no loss is shown.

Morning buyers are the most price-sensitive. In the afternoon the cart can charge \$3.50.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.